# KneeAxis — Complete Offline College GPU Training Automation
**AI-Assisted 3D Knee Reconstruction & Arthroplasty Planning**

Select **Kernel -> Restart Kernel and Run All Cells** to execute the complete pipeline sequentially on your college JupyterHub GPU server without needing internet access.

### CELL 1 — Detect project root

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path.cwd().resolve()
if (project_root / "src").is_dir():
    PROJECT_ROOT = project_root
elif (project_root.parent / "src").is_dir():
    PROJECT_ROOT = project_root.parent
else:
    PROJECT_ROOT = project_root

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root detected: {PROJECT_ROOT}")

### CELL 2 — Check Python and GPU

In [ ]:
import tensorflow as tf
print(f"Python Version:     {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Detected:      {len(gpus)}")
for i, gpu in enumerate(gpus):
    print(f"  • GPU {i}: {gpu.name}")
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"    - Dynamic VRAM memory growth enabled")
    except Exception as e:
        print(f"    - Memory growth note: {e}")

### CELL 3 — Check dependencies

In [ ]:
from environment_check import main as run_env_check
env_ok = run_env_check()
if env_ok != 0:
    print("⚠️  Some dependencies are missing. If offline, install from wheelhouse:")
    print("   pip install --no-index --find-links=wheelhouse -r requirements-college.txt")

### CELL 4 — Verify dataset folders

In [ ]:
from src.common.paths import load_config, get_dataset_paths
config = load_config()
ds_paths = get_dataset_paths(config)

print("Dataset Directory Check:")
for name, path in ds_paths.items():
    exists = "✅ Present" if path.is_dir() else "❌ Missing"
    print(f"  • {name.upper():16s}: {exists} ({path})")

### CELL 5 — Validate dataset manifests

In [ ]:
from scripts.prepare_manifests import build_all_manifests
from scripts.validate_datasets import validate_datasets

# Step 1: Discover files and build manifests
build_all_manifests(config)

# Step 2: Validate integrity & subject splitting
val_results = validate_datasets(config)

### CELL 6 — Display class and case counts

In [ ]:
clf_info = val_results.get("classification", {})
seg_info = val_results.get("segmentation", {})

print("\n--- Classification Data Summary ---")
print(f"Total Images:      {clf_info.get('total_records', 0)}")
print(f"Unique Subjects:   {clf_info.get('unique_subjects', 0)}")
print(f"Class Breakdown:   {clf_info.get('class_distribution', {})}")

print("\n--- Segmentation Data Summary ---")
print(f"Total 3D Cases:    {seg_info.get('total_cases', 0)}")
print(f"Labels Identified: {seg_info.get('label_values_found', [])}")

### CELL 7 — Run preflight

In [ ]:
from run_all import run_preflight, setup_master_logging
gpus = tf.config.list_physical_devices('GPU')
if not gpus and config.get('environment', {}).get('require_gpu', False):
    print('Notice: No physical GPU detected in current session. Overriding require_gpu=False for notebook execution.')
    config['environment']['require_gpu'] = False
logger = setup_master_logging()
preflight_ok = run_preflight(config, logger)
assert preflight_ok, "Preflight checks failed. Please address errors before training."

### CELL 8 — Run smoke test
Validates model architecture compilation, forward-backward pass, and atomic checkpoint saving.

In [ ]:
from run_all import run_smoke_test
smoke_ok = run_smoke_test(logger)
assert smoke_ok, "Technical smoke test failed."

### CELL 9 — Launch complete sequential training
Executes ResNet50 training, clears GPU memory, then executes 3D U-Net training.

In [ ]:
from run_all import run_subprocess_stage

# Stage 1: ResNet50 KL Classification
print("\n>>> Starting ResNet50 Classification Training...")
clf_cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "train_classification.py")]
clf_code = run_subprocess_stage(clf_cmd, PROJECT_ROOT / "logs" / "classification.log", "ResNet50 Training", logger)

# Stage 2: 3D U-Net Knee Segmentation
print("\n>>> Starting 3D U-Net Knee Segmentation Training...")
seg_cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "train_segmentation.py")]
seg_code = run_subprocess_stage(seg_cmd, PROJECT_ROOT / "logs" / "segmentation.log", "3D U-Net Training", logger)

### CELL 10 — Display training results
Evaluates test sets and displays performance metrics.

In [ ]:
# Run evaluations
eval_clf_cmd = [sys.executable, str(PROJECT_ROOT / "src" / "classification" / "evaluate.py"), "--dataset", "test"]
run_subprocess_stage(eval_clf_cmd, PROJECT_ROOT / "logs" / "evaluation.log", "Classification Evaluation", logger)

eval_seg_cmd = [sys.executable, str(PROJECT_ROOT / "src" / "segmentation" / "evaluate.py")]
run_subprocess_stage(eval_seg_cmd, PROJECT_ROOT / "logs" / "evaluation.log", "Segmentation Evaluation", logger)

### CELL 11 — Verify saved weights
Runs dummy-pass integrity checks and verifies JSON/NIfTI contracts.

In [ ]:
from scripts.verify_saved_models import main as run_verify
run_verify()

### CELL 12 — Generate final artifact ZIP

In [ ]:
from scripts.export_artifacts import export_artifacts
export_zip_path = export_artifacts()
print(f"\n✅ Final artifact package generated: {export_zip_path}")

### CELL 13 — Show the ZIP location and download instructions

In [ ]:
zip_file = PROJECT_ROOT / "KneeAxis_Trained_Artifacts.zip"
if zip_file.is_file():
    size_mb = zip_file.stat().st_size / (1024 * 1024)
    print("=" * 70)
    print(f"🎉 TRAINING & EXPORT COMPLETED SUCCESSFULLY!")
    print(f"File:     {zip_file.name}")
    print(f"Path:     {zip_file.resolve()}")
    print(f"Size:     {size_mb:.2f} MB")
    print("=" * 70)
    print("\nIntegration Instructions:")
    print("1. Download KneeAxis_Trained_Artifacts.zip via the JupyterHub file browser to your laptop.")
    print("2. Unzip and copy the folders into your KneeAxis application root:")
    print("   • models/kl_classifier/best.weights.h5")
    print("   • models/knee_segmentation/best.weights.h5")
    print("3. In KneeAxis models/model_integration_config.json, set 'use_trained_model': true")
    print("   (or toggle the switch in the web dashboard under AI Models -> Custom Training Space).")
else:
    print("⚠️  Artifact ZIP not found. Please review the logs in logs/master_training.log.")